In [1]:
# 02_counterfactual_sensitivity.ipynb
# Module 2: subgroup (SES x sex) counterfactual intervention sensitivity via positive,
# policy-deliverable engagement levers raised to the national weighted 75th percentile.
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
# --- grayscale publication style ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,
                     "axes.titlesize":1,"figure.autolayout":True})
GRAY = ["0.20","0.45","0.65","0.80"]
# --- resolve folders relative to repo (no path is printed, for anonymity) ---
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    # save both formats at 600 dpi, no caption/title; paths are never printed
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)

stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat"))
USED=["NISB","S_HOMLIT","S_INTNET","S_HISEI","S_EXCOMP","S_ICTSTUD","S_LRNINTS","S_LRNINTO",
      "S_ACMULT","S_GENEFF","S_SPECEFF","S_CTCLS","S_LRNSAFE","S_GENCLASS","S_SPECLASS","S_ICTPOSG","S_ICTFUT","S_ICTPOSS"]
for c in USED: stu.loc[(stu[c]>=995)|(stu[c]<=-995),c]=np.nan
stu.loc[(stu.NISB<-6)|(stu.NISB>6),"NISB"]=np.nan
# weighted SES tertiles x sex subgroups
w=stu.TOTWGTS.to_numpy(); x=stu.NISB.to_numpy(); ok=~np.isnan(x)
o=np.argsort(x[ok]); cw=np.cumsum(w[ok][o])/w[ok].sum(); q=np.interp([1/3,2/3],cw,x[ok][o])
stu["SES3"]=pd.Series(np.where(x<=q[0],"Low",np.where(x<=q[1],"Mid","High"))).where(ok).values
stu["SEX"]=stu.S_SEX.map({0.0:"Sex0",1.0:"Sex1"})
df=stu.dropna(subset=["SES3","SEX"]).copy()
# intervention levers: positive association + policy-deliverable (confounded in-class drills excluded, kept as controls)
LEVERS=["S_EXCOMP","S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]
CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT",
     "S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]
W=df.TOTWGTS.to_numpy(); Xb=df[LEVERS+CTX].to_numpy(float); Xc=Xb.copy()
for j,L in enumerate(LEVERS):
    v=df[L].to_numpy(); mm=~np.isnan(v); vs=np.sort(v[mm]); ww=W[mm][np.argsort(v[mm])]
    tgt=np.interp(0.75,np.cumsum(ww)/ww.sum(),vs); col=Xc[:,j]; Xc[:,j]=np.where(np.isnan(col),col,np.maximum(col,tgt))
dd=np.zeros((len(df),5))
for k,pv in enumerate(PVS):
    m=HistGradientBoostingRegressor(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,
        l2_regularization=1.0,random_state=0).fit(Xb,df[pv].to_numpy(float),sample_weight=W)
    dd[:,k]=m.predict(Xc)-m.predict(Xb)
df["dCIL"]=dd.mean(1)
rows=[]
for (s,x2),g in df.groupby(["SES3","SEX"]):
    ww=g.TOTWGTS.to_numpy()
    rows.append(dict(SES=s,SEX=x2,n=len(g),wN=ww.sum(),
                     base_CIL=np.average(g.PV1CIL,weights=ww),dCIL=np.average(g.dCIL,weights=ww)))
R=pd.DataFrame(rows).sort_values("dCIL",ascending=False)
R.to_csv(TAB/"m2_sensitivity.csv",index=False)
print(R.round(2).to_string(index=False))
print("corr(baseline, sensitivity) =", round(np.corrcoef(R.base_CIL,R.dCIL)[0,1],3))
# figure: grayscale scatter of baseline achievement vs intervention sensitivity (no title/caption)
fig,ax=plt.subplots(figsize=(4.2,3.4))
shade={"Low":"0.20","Mid":"0.50","High":"0.72"}
for _,r in R.iterrows():
    ax.scatter(r.base_CIL,r.dCIL,s=80,color=shade[r.SES],edgecolor="black",linewidth=0.6,zorder=3)
    ax.annotate(r.SES+"/"+r.SEX.replace("Sex","S"),(r.base_CIL,r.dCIL),fontsize=7,xytext=(4,3),textcoords="offset points")
z=np.polyfit(R.base_CIL,R.dCIL,1); xs=np.array([R.base_CIL.min(),R.base_CIL.max()])
ax.plot(xs,np.polyval(z,xs),color="0.35",ls=":",lw=1)
ax.set_xlabel("Baseline CIL"); ax.set_ylabel("Counterfactual sensitivity ($\\Delta$CIL)")
savefig(fig,"fig_m2_sensitivity")


 SES  SEX   n       wN  base_CIL  dCIL
 Low Sex0 632 67388.25    510.69 26.19
 Low Sex1 635 66465.84    534.42 23.39
 Mid Sex0 622 66773.73    530.02 21.40
High Sex0 619 69144.67    548.22 20.40
 Mid Sex1 593 66452.40    555.78 18.18
High Sex1 573 63917.32    579.92 15.48
corr(baseline, sensitivity) = -0.969
